# OutOfRole: feasibility study for the IBM Z Datathon 2026

**Team:** The Debugging Table (UT Dallas) · **Track:** AI Secured: Innovation Without Exposure · **Prepared:** Oct 16, 2026

**Question:** can we rank employees so that people misusing legitimate access (insiders) rise to the top, using only activity logs and no labels? And does a **recommender system** (implicit-feedback ALS: "access that is surprising for someone like you") add anything over simpler signals?

This notebook is the full record of the study, in the order it happened. Every step has:

1. **Why**: the reason for the step (a test result, a reviewer's critique, or published work)
2. **Code**: runnable, with outputs from the actual run
3. **Finding**: what the output shows and what we changed because of it

It is written for review, and has already been through three rounds of external review (Codex, gpt-6.1-sol). The fixes from that review are in this version; the review files are in `feasibility/codex_review/`. The questions we'd most like feedback on are listed in the last section.

### Results at a glance (after an external review and three rounds of fixes)

"found" = expected insiders in the top 70 (of 70 in r4.2, of 99 in r5.2), with ties ordered at random.

| Approach | r4.2: P@20 / found / AP | r5.2: P@20 / found / AP | On an unseen attack type |
|---|---|---|---|
| Baseline: 28-day personal window | 1.00 / 38 / 0.728 | 0.90 / 36 / 0.428 | spike attacks only |
| Own-history weekly percentile | 1.00 / 39.4 / 0.666 | 1.00 / 40.3 / 0.477 | best top-of-list on both releases |
| Label-free family-first Stouffer (frozen on r4.2) | 1.00 / 45 / 0.784 | 0.90 / 38 / 0.478 | no attack-label fitting |
| Trained combiner (logistic regression) | 1.00 / 51 to 52 / 0.857 | 0.90 / 49 / 0.630 | hidden job-hunters 5 to 16 of 30; r5.2 hidden leakers 0 to 28 of 29 |
| Gradient boosting | 1.00 / 60 / 0.940 | not run | hidden job-hunters 4 of 30 |
| Weekly persistence on unusual computers | no snoopers | 27 to 30 of 30 snoopers in the top 100 | designed after seeing r5.2 |
| **Equal budget, 100 people reviewed** | queues 42, fusion **55**, trained 63 | queues **53** (59 without the recommender queue), fusion 43, trained 55 | |

**In one paragraph:**
- Personal baselines and own-history weekly percentiles are the reliable core.
- Snoopers are caught by *persistence* (unusual computers week after week), not by the recommender or ownership as such.
- The recommender predicts normal browsing no better than popularity and adds only modest, conditional value.
- Trained combiners look best in cross-validation but generalize poorly to unseen attacks.
- The best label-free design depends on the threat mix: fusion on r4.2, specialist queues on r5.2.
- Job-hunters stay hard.

## Protocol (fixed before any result was seen)

| Item | Choice | Reason |
|---|---|---|
| Data | CMU SEI CERT Insider Threat Test Dataset, releases **r4.2** (1,000 staff, 70 insiders, scenarios 1-3) and **r5.2** (2,000 staff, 99 insiders, adds scenario 4) | Public, labeled, standard benchmark; r5.2 tests generalization to a new attack type |
| Unit of evaluation | **One score per person**; all staff ranked | Analysts review people, not log lines; event-level metrics inflate results |
| Train window | r4.2: before **2010-06-01**; r5.2: before **2010-04-01** | No insider activity starts before these dates (checked in Step 2), so training cannot see attacks |
| Labels | Used **only to evaluate**, except where a trained combiner is explicitly marked and cross-validated | Real deployments rarely have labels |
| Never used | The LDAP "left the company" signal | Nearly every insider leaves (all 70 in r4.2); it is hindsight, not detection |
| Weekly features | The week containing the cutoff is dropped from both train and test | It mixes train and test days |
| Ties | Top-k counts are **expected values** when tied scores are ordered at random | Integer-valued signals tie often; no luck from user-ID order |
| Exploratory look-ahead | The after-hours window (before 7:00, from 19:00) was read off the whole-log login histogram | Uses no labels, but is a design choice informed by all data |
| Metrics | **P@20** (share of the top 20 who are insiders), **hits@70** (insiders in the top 70), **AP** (average precision), **AUC**, and **insiders of each scenario in the top 100** | Top-of-list precision is what an analyst feels; AP/AUC summarize the whole ranking |

**The scenarios** (from the dataset's answer key):

1. **Leaker:** starts working after hours, uses a USB drive for the first time, uploads to wikileaks.org, leaves.
2. **Job-hunter thief:** browses job and competitor sites, then copies files to USB far more than usual, leaves.
3. **Disgruntled sysadmin:** downloads a keylogger, plants it on the supervisor's PC by USB, logs in as the supervisor, sends a mass email, leaves.
4. **Snooper (r5.2 only):** logs into other people's computers, searches files and emails them home, more and more often over 3 months.

**Environment:** laptop prototype (12 cores, 36 GB RAM), Spark 3.5.9 in local mode, Python 3.11, scikit-learn. The datathon build runs on an IBM LinuxONE server; platform notes are at the end.

In [1]:
import os, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm, rankdata
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.recommendation import ALS

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)

BASE = Path.cwd().parent if Path.cwd().name == "feasibility" else Path.cwd()
DATA = BASE / "data"
ANS = DATA / "answers" / "answers"
WORK = BASE / "feasibility" / "work"; WORK.mkdir(parents=True, exist_ok=True)
REL = {"4.2": {"raw": DATA / "r4.2" / "r4.2", "cut": pd.Timestamp("2010-06-01")},
       "5.2": {"raw": DATA / "r5.2" / "r5.2", "cut": pd.Timestamp("2010-04-01")}}
SEED = 7
T0 = time.time()

spark = (SparkSession.builder.master("local[10]").appName("outofrole-feasibility")
         .config("spark.driver.memory", "16g").config("spark.sql.shuffle.partitions", "64")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| data:", DATA)

26/10/16 22:55:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.9 | data: /Users/mehrad/Documents/School/UTD/UTD_as_student/Fall2026/IBM_Z_Datathon_2026/data


In [2]:
# ---------- labels and evaluation helpers ----------
INS = pd.read_csv(ANS / "insiders.csv")

def labels(rel):
    d = INS[INS.dataset.astype(str) == rel]
    return dict(zip(d.user, d.scenario))

def aligned(s, c):
    # arrays from fusion code are already in c.users order; Series are aligned by person ID; missing = no evidence
    s = pd.Series(s).reindex(c.users) if isinstance(s, pd.Series) else pd.Series(np.asarray(s, float), index=c.users)
    return s.fillna(min(0.0, s.min()) if s.notna().any() else 0.0)

def top_expected(v, mask, k):
    # expected number of `mask` people in the top k when tied scores are ordered at random (no ID-order luck)
    thr = np.sort(v)[::-1][k - 1]; above, tie = v > thr, v == thr
    return mask[above].sum() + (k - above.sum()) * mask[tie].sum() / tie.sum()

def fmt(e, n):
    return f"{e:.0f}/{n}" if abs(e - round(e)) < 0.05 else f"{e:.1f}/{n}"

def metrics(name, s, c, k_top=100):
    s = aligned(s, c); v = s.values.astype(float); y = np.array([u in c.lab for u in c.users])
    row = {"detector": name, "P@20": top_expected(v, y, 20) / 20, "hits@70": round(top_expected(v, y, 70), 1),
           "AP": average_precision_score(y, v), "AUC": roc_auc_score(y, v)}
    for sc in sorted(set(c.lab.values())):
        m = np.array([c.lab.get(u) == sc for u in c.users])
        row[f"S{sc} top{k_top}"] = fmt(top_expected(v, m, k_top), int(m.sum()))
    return row

def table(rows):
    return pd.DataFrame(rows).set_index("detector").round(3)

def pad_days(c, d, cols):
    # every active test day of every person, with 0 where that day had no novelty
    base = c.te[["user", "day"]].drop_duplicates()
    return base.merge(d, on=["user", "day"], how="left").fillna({k: 0.0 for k in cols})

def robust_max(df, col):
    # per person: how far their worst day sits above their own typical active test day (robust z).
    # Retrospective by design: the typical day is taken over the whole test period.
    g = df.groupby("user")[col]
    med = g.transform("median"); mad = g.transform(lambda x: (x - x.median()).abs().median())
    return ((df[col] - med) / (1.4826 * mad + 1)).groupby(df.user).max()

def rz(s, users):
    # robust z across people on log scale; mostly-zero signals fall back to the standard deviation
    v = np.log1p(pd.Series(s).reindex(users).fillna(0).clip(lower=0))
    mad = 1.4826 * (v - v.median()).abs().median()
    scale = mad if mad > 0 else v.std()
    return (v - v.median()) / (scale if scale > 0 else 1.0)

def pct(s, users):
    s = pd.Series(s).reindex(users)
    return s.fillna(s.min()).rank(pct=True)

## Step 1: Load the raw logs with Spark

**Why.** The four logs we need (logon, USB device, file copies, web) total about 15 GB for r4.2 and 37 GB for r5.2, mostly the web log. pandas would load whole files into memory; Spark streams them in partitions and keeps only the columns we ask for. Local mode uses every core of one machine, no cluster needed.

**What we keep.**
- Per person per day: logins, after-hours logins, USB connects, file copies, web visits (and their after-hours counts), distinct computers.
- Person-to-computer pairs (one row per person, day, computer logged into).
- Person-to-website pairs (URL reduced to its site, e.g. `wikileaks.org`, with a visit count).

"After hours" is before 7:00 or from 19:00, chosen from the logon-hour histogram below. In r5.2 the file log records every file action, so we keep only copies to removable media, which matches what r4.2's file log contains.

In [3]:
def etl(rel, force=False):
    out, raw = WORK / f"r{rel}", REL[rel]["raw"]
    if (out / "daily").exists() and not force:
        return out
    out.mkdir(parents=True, exist_ok=True)
    def load(name, cols):
        df = spark.read.csv(str(raw / f"{name}.csv"), header=True).select(*cols)
        ts = F.to_timestamp("date", "MM/dd/yyyy HH:mm:ss")
        return df.withColumn("ts", ts).withColumn("day", F.to_date("ts")).withColumn("hour", F.hour("ts")).drop("date")
    logon = load("logon", ["date", "user", "pc", "activity"]).filter(F.col("activity") == "Logon")
    device = load("device", ["date", "user", "pc", "activity"]).filter(F.col("activity") == "Connect")
    if "to_removable_media" in open(raw / "file.csv").readline():
        files = load("file", ["date", "user", "pc", "to_removable_media"]).filter(F.col("to_removable_media") == "True").drop("to_removable_media")
    else:
        files = load("file", ["date", "user", "pc"])
    http = (load("http", ["date", "user", "pc", "url"])
            .withColumn("domain", F.lower(F.regexp_extract("url", r"^https?://(?:www\.)?([^/:]+)", 1))).drop("url"))
    logon.groupBy("hour").count().orderBy("hour").toPandas().to_csv(out / "logon_hours.csv", index=False)
    ah = lambda: ((F.col("hour") < 7) | (F.col("hour") >= 19)).cast("int")
    d1 = logon.groupBy("user", "day").agg(F.count("*").alias("n_logon"), F.sum(ah()).alias("n_logon_ah"), F.countDistinct("pc").alias("n_pcs"))
    d2 = device.groupBy("user", "day").agg(F.count("*").alias("n_usb"), F.sum(ah()).alias("n_usb_ah"))
    d3 = files.groupBy("user", "day").agg(F.count("*").alias("n_file"), F.sum(ah()).alias("n_file_ah"))
    d4 = http.groupBy("user", "day").agg(F.count("*").alias("n_http"), F.sum(ah()).alias("n_http_ah"))
    (d1.join(d2, ["user", "day"], "full").join(d3, ["user", "day"], "full").join(d4, ["user", "day"], "full")
       .fillna(0).write.mode("overwrite").parquet(str(out / "daily")))
    logon.select("user", "day", "pc").distinct().write.mode("overwrite").parquet(str(out / "pc_pairs"))
    (http.filter(F.col("domain") != "").groupBy("user", "day", "domain").agg(F.count("*").alias("n"))
         .write.mode("overwrite").parquet(str(out / "dom_pairs")))
    return out

class Ctx:  # everything about one release
    pass

def context(rel):
    t = time.time(); out = etl(rel)
    c = Ctx(); c.rel, c.cut, c.raw, c.lab = rel, REL[rel]["cut"], REL[rel]["raw"], labels(rel)
    c.daily = pd.read_parquet(out / "daily"); c.daily["day"] = pd.to_datetime(c.daily["day"])
    c.pcs = pd.read_parquet(out / "pc_pairs"); c.pcs["day"] = pd.to_datetime(c.pcs["day"])
    c.dom = pd.read_parquet(out / "dom_pairs"); c.dom["day"] = pd.to_datetime(c.dom["day"])
    c.users = sorted(c.daily.user.unique()); c.uidx = {u: i for i, u in enumerate(c.users)}
    c.tr, c.te = c.daily[c.daily.day < c.cut], c.daily[c.daily.day >= c.cut].copy()
    c.hours = pd.read_csv(out / "logon_hours.csv")
    c.S = {}
    print(f"r{rel}: loaded in {time.time() - t:.0f}s | staff {len(c.users)} | insiders {len(c.lab)} | days {c.daily.day.nunique()} "
          f"| train days {c.tr.day.nunique()} | test days {c.te.day.nunique()}")
    return c

c42 = context("4.2")
print("totals:", c42.daily[["n_logon", "n_usb", "n_file", "n_http"]].sum().to_dict())
print("distinct computers:", c42.pcs.pc.nunique(), "| distinct websites:", c42.dom.domain.nunique())
print("logons by hour:", dict(zip(c42.hours.hour, c42.hours["count"])))

r4.2: loaded in 0s | staff 1000 | insiders 70 | days 501 | train days 150 | test days 351
totals: {'n_logon': 470591, 'n_usb': 203339, 'n_file': 445581, 'n_http': 28434423}
distinct computers: 1003 | distinct websites: 475
logons by hour: {0: 2347, 1: 2317, 2: 2206, 3: 2263, 4: 2244, 5: 2408, 6: 3954, 7: 146662, 8: 139629, 9: 49992, 10: 7734, 11: 16702, 12: 28151, 13: 28952, 14: 11838, 15: 2332, 16: 1880, 17: 2271, 18: 2727, 19: 2783, 20: 2810, 21: 2856, 22: 3094, 23: 2439}


**Finding.** Spark turns r4.2's raw logs into three small tables (about 36 seconds on a cold laptop run; the run shown here reads the cache): 1,000 staff over 501 days (150 train, 351 test), 470,591 logins, 203,339 USB connects, 445,581 file copies and 28.4 million web visits.

- The catalog is small: 1,003 computers and 475 websites, so a recommender has few items to model.
- Logins cluster at 7:00 to 9:00 and 11:00 to 14:00. Each hour before 7:00 or from 19:00 has about 2,000 to 4,000 logins, which sets the after-hours window. That window was read off the whole log, which is an exploratory look-ahead, though label-free.

**Change:** the web log becomes a core input.

## Step 2: Check for leakage before building anything

**Why.** A reviewer's first critique was that a time split can leak the answers if training data contains attacks, and that some fields give away the label. We checked both directly in the answer key and the org chart.

In [4]:
rows = []
for rel in ("4.2", "5.2"):
    d = INS[INS.dataset.astype(str) == rel].copy(); d["start"] = pd.to_datetime(d.start)
    for sc, g in d.groupby("scenario"):
        rows.append({"release": rel, "scenario": sc, "insiders": len(g), "first malicious event": g.start.min().date(), "train window ends": REL[rel]["cut"].date()})
display(pd.DataFrame(rows))

for rel in ("4.2", "5.2"):
    snaps = {f.name[:7]: set(pd.read_csv(f).user_id) for f in sorted((REL[rel]["raw"] / "LDAP").glob("*.csv"))}
    first, last = snaps[min(snaps)], snaps[max(snaps)]
    left = first - last; lab = labels(rel)
    print(f"r{rel}: staff at start {len(first)}, at end {len(last)}, new hires {len(set().union(*snaps.values()) - first)}, "
          f"left {len(left)} of whom insiders {len(left & set(lab))} of {len(lab)}")

,release,scenario,insiders,first malicious event,train window ends
0,4.2,1,30,2010-07-07,2010-06-01
1,4.2,2,30,2010-06-14,2010-06-01
2,4.2,3,10,2010-06-10,2010-06-01
3,5.2,1,29,2010-07-07,2010-04-01
4,5.2,2,30,2010-06-23,2010-04-01
5,5.2,3,10,2010-06-17,2010-04-01
6,5.2,4,30,2010-04-16,2010-04-01


r4.2: staff at start 1000, at end 845, new hires 0, left 155 of whom insiders 70 of 70
r5.2: staff at start 2000, at end 1762, new hires 0, left 238 of whom insiders 70 of 99


**Finding.** No attack is in training, and one trap must be avoided.

- **Train windows:** the first malicious event is June 10, 2010 (r4.2) and April 16, 2010 (r5.2), both after the train cutoffs. Weekly features drop the week that contains the cutoff.
- **Departures:** all 70 r4.2 insiders leave, while only 155 of 1,000 staff leave in total. In r5.2, 70 of the 99 insiders leave; the 29 who stay are all snoopers (one snooper left). "Left the company" is hindsight and is never used.
- **New hires:** none in either release, so cold start can't be tested here.

## Step 3: Baselines a security analyst would build

**Why.** A model only matters if it beats simple, explainable rules. Three baselines, all label-free:

- **R0 rule:** number of days with after-hours USB use.
- **R1 daily behavior z-score:** for each count, how many standard deviations today is above the person's own train-window average; summed over counts; the person's worst day.
- **R2 28-day window:** the same idea over rolling 28-day totals against the person's own train-window rate, to catch slow ramps.

In [5]:
FEATS = ["n_logon_ah", "n_usb", "n_usb_ah", "n_file", "n_file_ah", "n_http_ah", "n_pcs"]
WF = ["n_usb", "n_file", "n_logon_ah", "n_usb_ah", "n_http_ah"]

def window_z(c, w):
    rate = c.tr.groupby("user")[WF].sum() / c.tr.groupby("user").day.nunique().values[:, None]
    te_s = c.te.sort_values(["user", "day"]).set_index("day")
    roll = te_s.groupby("user")[WF].rolling(f"{w}D").sum().reset_index()
    nd = te_s.groupby("user")["n_logon"].rolling(f"{w}D").count().reset_index(drop=True)
    exp = roll[["user"]].join(rate, on="user")[WF].fillna(0).values * nd.values[:, None]
    roll["z"] = np.clip((roll[WF].values - exp) / np.sqrt(exp + 1), 0, None).sum(axis=1)
    return roll, roll.groupby("user").z.max()

def det_baselines(c):
    te = c.te.copy()
    c.S["r0"] = te.assign(f=te.n_usb_ah > 0).groupby("user").f.sum()
    mu, sd = c.tr.groupby("user")[FEATS].mean(), c.tr.groupby("user")[FEATS].std().fillna(0)
    m = te[["user"]].join(mu, on="user")[FEATS].fillna(0).values
    s = te[["user"]].join(sd, on="user")[FEATS].fillna(0).values
    te["r1"] = np.clip((te[FEATS].values - m) / (s + 1), 0, None).sum(axis=1)
    c.te_r1, c.mu, c.sd = te[["user", "day", "r1"]], mu, sd
    c.S["r1"] = te.groupby("user").r1.max()
    _, c.S["r2"] = window_z(c, 28)

det_baselines(c42)
display(table([metrics("R0 rule: days with after-hours USB", c42.S["r0"], c42),
               metrics("R1 daily behavior z", c42.S["r1"], c42),
               metrics("R2 28-day window z", c42.S["r2"], c42)]))

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
R0 rule: days with after-hours USB,0.3,33.5,0.272,0.769,30/30,4/30,8/10
R1 daily behavior z,1.0,38.0,0.680,0.943,30/30,2/30,10/10
R2 28-day window z,1.0,38.0,0.728,0.960,30/30,5/30,9/10


**Finding.** Personal baselines are strong, and they have a clear blind spot.

- R1 and R2 reach P@20 1.00, 38 of 70 in the top 70, and AUC 0.94 to 0.96. Every leaker is in the top 100. The hand rule reaches P@20 0.30.
- Only 2 (daily) and 5 (28-day) of the 30 job-hunters reach the top 100.

**Why more is needed:** these baselines see spikes, not slow drift, and not which resources someone touches.

## Step 4: Novelty, naive versus recommender (ALS)

**Why.** The project's core idea. For each test-period login or web visit to something the person never used in training, ask how surprising it is:

- **Naive novelty:** every new computer or site counts 1.
- **ALS novelty:** Spark MLlib ALS with implicit feedback (from CS 6350 Lab 3), trained on train-window person-to-computer and person-to-site counts (log-scaled), rank 16, alpha 10, regularization 0.1. Surprise = 1 minus predicted affinity, so a new site that similar people already use is forgiven. Sites nobody visited in training get surprise 1.
- **Rare sites:** visits to sites that under 1% of staff visited in training.

For each person and active day we add up the **distinct** new computers or sites that day (not visit counts). Each person is scored by how far their worst day sits above their typical active day; days with no novelty count as zero.

In [6]:
def split_new(c, pairs, item):
    trp, tep = pairs[pairs.day < c.cut], pairs[pairs.day >= c.cut]
    seen = set(zip(trp.user, trp[item]))
    return trp, tep[[(u, v) not in seen for u, v in zip(tep.user, tep[item])]].copy()

def train_als(c, trp, item, rank=16, alpha=10.0, reg=0.1, seed=SEED):
    cnt = (trp.groupby(["user", item]).n.sum() if "n" in trp else trp.groupby(["user", item]).size()).rename("c").reset_index()
    items = sorted(cnt[item].unique()); iidx = {v: i for i, v in enumerate(items)}
    sdf = spark.createDataFrame(pd.DataFrame({"u": cnt.user.map(c.uidx).astype(int), "i": cnt[item].map(iidx).astype(int),
                                              "r": np.log1p(cnt.c).astype(float)}))
    m = ALS(userCol="u", itemCol="i", ratingCol="r", implicitPrefs=True, alpha=alpha, rank=rank,
            regParam=reg, maxIter=10, seed=seed).fit(sdf)
    U = np.zeros((len(c.users), rank)); V = np.zeros((len(items), rank))
    for r in m.userFactors.collect(): U[r.id] = r.features
    for r in m.itemFactors.collect(): V[r.id] = r.features
    return U, V, iidx, cnt

def predict(c, U, V, iidx, df, item):
    ii = df[item].map(iidx); known = ii.notna().values; pred = np.zeros(len(df))
    pred[known] = np.einsum("ij,ij->i", U[df.user.map(c.uidx).values[known]], V[ii[known].astype(int).values])
    return pred

def det_novelty(c, seed=SEED):
    pc_tr, pc_new = split_new(c, c.pcs, "pc"); dom_tr, dom_new = split_new(c, c.dom, "domain")
    for tr_, new_, item in ((pc_tr, pc_new, "pc"), (dom_tr, dom_new, "domain")):
        U, V, iidx, _ = train_als(c, tr_, item, seed=seed)
        new_["s"] = 1 - np.clip(predict(c, U, V, iidx, new_, item), 0, 1)
        new_["one"] = 1.0
    def scores(col):
        d = pd.merge(pc_new.groupby(["user", "day"])[col].sum().rename("pc").reset_index(),
                     dom_new.groupby(["user", "day"])[col].sum().rename("dom").reset_index(), on=["user", "day"], how="outer").fillna(0)
        d = pad_days(c, d, ["pc", "dom"])
        return robust_max(d, "pc"), robust_max(d, "dom")
    c.S["n_pc"], c.S["n_dom"] = scores("one")
    c.S["a_pc"], c.S["a_dom"] = scores("s")
    pop = dom_tr.groupby("domain").user.nunique() / len(c.users)
    dte = c.dom[c.dom.day >= c.cut]
    rare = set(pop[pop < 0.01].index) | (set(dte.domain) - set(pop.index))
    c.S["w_rare"] = dte[dte.domain.isin(rare)].groupby("user").n.sum()
    c.pc_new, c.dom_new, c.pc_tr, c.dom_tr = pc_new, dom_new, pc_tr, dom_tr

det_novelty(c42)
S = c42.S
print(f"new test pairs: computers {len(c42.pc_new):,}, websites {len(c42.dom_new):,}")
display(table([metrics("N1 naive: new computers", S["n_pc"], c42), metrics("N2 naive: new websites", S["n_dom"], c42),
               metrics("A1 ALS: surprising computers", S["a_pc"], c42), metrics("A2 ALS: surprising websites", S["a_dom"], c42),
               metrics("N  naive (computers + sites)", pct(S["n_pc"], c42.users) + pct(S["n_dom"], c42.users), c42),
               metrics("A  ALS (computers + sites)", pct(S["a_pc"], c42.users) + pct(S["a_dom"], c42.users), c42),
               metrics("W1 visits to rare sites (<1% of staff)", S["w_rare"], c42)]))

dn = c42.dom_new
hot = dn[dn.user.isin(c42.lab)].groupby("domain").agg(insiders=("user", "nunique"), mean_surprise=("s", "mean"))
hot = hot.join(dn[~dn.user.isin(c42.lab)].groupby("domain").user.nunique().rename("normal_staff")).fillna(0)
print("New sites visited by the most insiders in the test window:")
display(hot[hot.mean_surprise > 0.9].sort_values("insiders", ascending=False).head(8))

new test pairs: computers 16,423, websites 195,078


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
N1 naive: new computers,0.231,11.7,0.101,0.579,4.6/30,2.4/30,7/10
N2 naive: new websites,0.050,2.0,0.099,0.594,3/30,4.2/30,0/10
A1 ALS: surprising computers,0.150,10.3,0.099,0.578,4/30,3/30,8/10
A2 ALS: surprising websites,0.000,5.0,0.122,0.713,1/30,8/30,4/10
N naive (computers + sites),0.200,13.0,0.126,0.634,3/30,3/30,8/10
A ALS (computers + sites),0.250,17.0,0.164,0.733,4.5/30,3/30,10/10
W1 visits to rare sites (<1% of staff),0.496,41.0,0.391,0.779,30/30,1.9/30,10/10


New sites visited by the most insiders in the test window:


,insiders,mean_surprise,normal_staff
domain,,,
wikileaks.org,30,1.0,0.0
refog.com,7,1.0,0.0
download.cnet.com,5,1.0,0.0
relytec.com,5,1.0,0.0
wellresearchedreviews.com,5,1.0,0.0
actualkeylogger.com,4,1.0,0.0
dailykeylogger.com,4,1.0,0.0
softactivity.com,4,1.0,0.0


**Finding.** Alone, novelty is weak, with one exception.

- **Rare-site visits** reach P@20 0.50 and 41 of 70 (AUC 0.78). In the test window, wikileaks.org shows up as new for 30 insiders and no normal staff. A separate check over the entire log found exactly 30 users ever visited it, all leakers.
- **Recommender vs naive:** the recommender's website surprise ranks the whole list better than the naive "new site" count (AUC 0.713 vs 0.594), but its top of the list is poor (P@20 0.00) once every active day counts.

**Change:** test the recommender as an addition to other signals, not as a detector.

## Step 5: First combinations, and does ALS earn its place?

**Why.** A reviewer's central challenge: "an access ALS would not recommend is not automatically an anomaly; show that low recommendation scores actually correlate with malicious behavior." We combined the behavior baselines with novelty in three ways and ran an ablation: the same combination with naive novelty instead of ALS.

- **Rank sum:** add each signal's percentile rank (label-free).
- **Robust-z sum:** add each signal's robust z-score (label-free).
- **Learned combiner:** logistic regression over all signal scores, 5-fold stratified cross-validation repeated over 5 shuffles (uses labels; every person is scored by a model that never saw them).

In [7]:
def Xmat(c, cols):
    X = pd.DataFrame({k: pd.Series(c.S[k]).reindex(c.users) for k in cols}).fillna(0)
    return np.log1p(X.clip(lower=0))

def make_est(kind, s):
    if kind == "lr":
        return make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=3000))
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, class_weight="balanced", random_state=s)

def cv_score(c, cols, kind="lr", seeds=5):
    X = Xmat(c, cols); y = np.array([u in c.lab for u in c.users])
    ps = [cross_val_predict(make_est(kind, s), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=s), method="predict_proba")[:, 1]
          for s in range(seeds)]
    return pd.Series(np.mean(ps, axis=0), index=c.users)

U42 = c42.users
rows = [metrics("rank sum: R1 + naive novelty", pct(S["r1"], U42) + pct(S["n_pc"], U42) + pct(S["n_dom"], U42), c42),
        metrics("rank sum: R1 + ALS novelty", pct(S["r1"], U42) + pct(S["a_pc"], U42) + pct(S["a_dom"], U42), c42),
        metrics("robust-z sum: R1 + R2 + rare sites", rz(S["r1"], U42) + rz(S["r2"], U42) + rz(S["w_rare"], U42), c42)]
B0 = ["r1", "r2", "w_rare"]
rows += [metrics("learned (LR): behavior + rare sites", cv_score(c42, B0), c42),
         metrics("learned (LR): + naive novelty", cv_score(c42, B0 + ["n_pc", "n_dom"]), c42),
         metrics("learned (LR): + ALS novelty", cv_score(c42, B0 + ["a_pc", "a_dom"]), c42)]
display(table(rows))

# is the ALS gain luck? retrain ALS with other random seeds
seed_rows = []
for sd_ in (11, 23):
    tmp = context("4.2"); det_baselines(tmp); det_novelty(tmp, seed=sd_)
    seed_rows.append(metrics(f"learned (LR): + ALS novelty, ALS seed {sd_}", cv_score(tmp, B0 + ["a_pc", "a_dom"]), tmp))
display(table(seed_rows))

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
rank sum: R1 + naive novelty,0.55,18.0,0.320,0.845,6/30,8/30,9/10
rank sum: R1 + ALS novelty,0.55,22.0,0.396,0.896,10/30,8/30,10/10
robust-z sum: R1 + R2 + rare sites,1.00,41.0,0.715,0.947,30/30,2/30,10/10
learned (LR): behavior + rare sites,1.00,39.0,0.733,0.961,30/30,6/30,10/10
learned (LR): + naive novelty,1.00,41.0,0.759,0.966,29/30,11/30,9/10
learned (LR): + ALS novelty,1.00,45.0,0.785,0.971,30/30,14/30,10/10


r4.2: loaded in 0s | staff 1000 | insiders 70 | days 501 | train days 150 | test days 351


r4.2: loaded in 0s | staff 1000 | insiders 70 | days 501 | train days 150 | test days 351


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
"learned (LR): + ALS novelty, ALS seed 11",1.0,45.0,0.778,0.970,30/30,13/30,10/10
"learned (LR): + ALS novelty, ALS seed 23",1.0,44.0,0.782,0.972,29/30,15/30,10/10


**Finding.**

- **Rank sums dilute** the baseline (P@20 0.55). A robust-z sum keeps P@20 1.00 (41 of 70).
- **In the learned combiner:**
  - behavior + rare sites: 39
  - + naive novelty: 41
  - + recommender: **45** (45 and 44 with ALS seeds 11 and 23)
  - AP: 0.733, then 0.759, then 0.785

**Reading:** on r4.2, the recommender adds about 4 insiders beyond the naive check. This is a net count, and Step 15 tests it more strictly.

## Step 6: More signals from the critique (peers, time windows, a second model)

**Why.** The critique raised cases a per-person baseline gets wrong: someone covering a colleague's duties, or a team-wide change such as a deadline crunch. We added:

- **Peer signals:** each person against their **team** that same day, and their rolling 28-day totals against the team's median 28-day total (teams from the org chart, median 21 people).
- **More windows:** 7 and 56 days, the mean of the 3 worst days, and the number of clearly odd days.
- **Isolation Forest:** a second, model-based anomaly detector over each day's personal z-vector, fitted on train days.

Then a richer learned combiner, a gradient-boosting combiner, and an ALS ablation inside each.

In [8]:
def det_more(c):
    team_df = pd.read_csv(c.raw / "LDAP" / "2010-01.csv")
    team = dict(zip(team_df.user_id, team_df.functional_unit + "|" + team_df.department + "|" + team_df.team))
    g = c.te_r1.groupby("user").r1
    c.S["r1_top3"] = g.apply(lambda x: x.nlargest(3).mean())
    c.S["r1_days5"] = g.apply(lambda x: (x > 5).sum())
    te = c.te.copy(); te["team"] = te.user.map(team)
    pm = te.groupby(["team", "day"])[FEATS].transform("mean").values
    ps = te.groupby(["team", "day"])[FEATS].transform("std").fillna(0).values
    te["peer"] = np.clip((te[FEATS].values - pm) / (ps + 1), 0, None).sum(axis=1)
    c.S["peer_daily"] = te.groupby("user").peer.max()
    _, c.S["r_w7"] = window_z(c, 7)
    roll, c.S["r_w56"] = window_z(c, 56)
    roll28, _ = window_z(c, 28); roll28["team"] = roll28.user.map(team)
    tm = roll28.groupby(["team", "day"])[WF].transform("median").values
    roll28["pz"] = np.clip((roll28[WF].values - tm) / np.sqrt(tm + 1), 0, None).sum(axis=1)
    c.S["peer_w28"] = roll28.groupby("user").pz.max()
    ztr = np.clip((c.tr[FEATS].values - c.tr[["user"]].join(c.mu, on="user")[FEATS].values)
                  / (c.tr[["user"]].join(c.sd, on="user")[FEATS].values + 1), 0, None)
    zte = np.clip((c.te[FEATS].values - c.te[["user"]].join(c.mu, on="user")[FEATS].fillna(0).values)
                  / (c.te[["user"]].join(c.sd, on="user")[FEATS].fillna(0).values + 1), 0, None)
    iso = IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1).fit(ztr)
    c.S["iforest"] = pd.Series(-iso.score_samples(zte), index=c.te.user.values).groupby(level=0).max()
    c.team = team

det_more(c42)
display(table([metrics(f"single: {k}", S[k], c42) for k in ["peer_daily", "peer_w28", "r_w7", "r_w56", "r1_top3", "r1_days5", "iforest"]]))

ALS3 = ["a_pc", "a_dom"]
L6 = ["r1", "r2", "w_rare", "peer_daily", "peer_w28", "r_w7", "r_w56", "r1_top3", "r1_days5", "iforest"]
display(table([metrics("LR: all signals, no ALS", cv_score(c42, L6), c42),
               metrics("LR: all signals + ALS", cv_score(c42, L6 + ALS3), c42),
               metrics("GBT: all signals, no ALS", cv_score(c42, L6, "gbt"), c42),
               metrics("GBT: all signals + ALS", cv_score(c42, L6 + ALS3, "gbt"), c42)]))

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
single: peer_daily,0.700,27.0,0.427,0.856,24/30,4/30,4/10
single: peer_w28,0.100,8.0,0.194,0.824,6/30,2/30,4/10
single: r_w7,1.000,37.0,0.672,0.942,30/30,0/30,10/10
single: r_w56,1.000,39.0,0.704,0.947,30/30,2/30,9/10
single: r1_top3,1.000,39.0,0.687,0.948,29/30,2/30,10/10
single: r1_days5,0.183,24.2,0.271,0.838,26.8/30,4.4/30,5.1/10
single: iforest,0.950,37.0,0.620,0.937,30/30,4/30,10/10


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
"LR: all signals, no ALS",1.0,45.0,0.782,0.969,30/30,16/30,9/10
LR: all signals + ALS,1.0,49.0,0.817,0.976,30/30,19/30,9/10
"GBT: all signals, no ALS",1.0,60.0,0.936,0.992,30/30,25/30,10/10
GBT: all signals + ALS,1.0,60.0,0.940,0.993,30/30,24/30,10/10


**Finding.**

- **Peers and windows:** team peer comparison reaches P@20 0.70. The extra windows and the Isolation Forest each reach P@20 0.95 to 1.00 alone.
- **Logistic regression:** 45 of 70 without ALS, 49 with it.
- **Gradient boosting:** 60 of 70 (AP 0.94) with or without ALS. Step 7 tests whether that is real.

## Step 7: Stress test: can it catch an attack type it never saw?

**Why.** Gradient boosting's score looked too good. The job-hunters are synthetic and share a pattern, so a trained model may simply memorize "what scenario 2 looks like". Real attackers invent new methods. **Leave one scenario out:** train the combiner without any insider of scenario *k*, then count how many scenario-*k* insiders reach the top 100 among normal staff. The label-free robust-z sum needs no training and is shown for reference.

In [9]:
def loso(c, cols, kind="lr", seeds=5):
    X = Xmat(c, cols).values; sc = np.array([c.lab.get(u, 0) for u in c.users]); y = sc > 0; res = {}
    for held in sorted(set(sc) - {0}):
        p = np.zeros(len(X)); tst = sc == held; pool = np.where(~tst)[0]
        for s in range(seeds):
            held_idx = np.where(tst)[0].copy(); np.random.RandomState(s).shuffle(held_idx)
            held_folds = np.array_split(held_idx, 5)          # held-out insiders join the outer folds too
            for f, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=s).split(pool, y[pool])):
                m = make_est(kind, s).fit(X[pool[a]], y[pool[a]])
                p[pool[b]] += m.predict_proba(X[pool[b]])[:, 1] / seeds
                if len(held_folds[f]): p[held_folds[f]] += m.predict_proba(X[held_folds[f]])[:, 1] / seeds
        keep = (sc == 0) | tst
        res[f"S{held} hidden"] = fmt(top_expected(p[keep], sc[keep] == held, 100), int(tst.sum()))
    return res

def loso_unsup(c, cols):
    p = sum(rz(c.S[k], c.users) for k in cols).values
    sc = np.array([c.lab.get(u, 0) for u in c.users]); res = {}
    for held in sorted(set(sc) - {0}):
        keep = (sc == 0) | (sc == held)
        res[f"S{held} hidden"] = fmt(top_expected(p[keep], sc[keep] == held, 100), int((sc == held).sum()))
    return res

rows = []
for name, cols, kind in (("LR, all signals + ALS", L6 + ALS3, "lr"), ("LR, no ALS", L6, "lr"),
                         ("GBT, all signals + ALS", L6 + ALS3, "gbt"), ("GBT, no ALS", L6, "gbt")):
    rows.append({"combiner": name, **loso(c42, cols, kind)})
rows.append({"combiner": "no training: robust-z sum, all signals + ALS", **loso_unsup(c42, L6 + ALS3)})
display(pd.DataFrame(rows).set_index("combiner"))

,S1 hidden,S2 hidden,S3 hidden
combiner,,,
"LR, all signals + ALS",30/30,5/30,10/10
"LR, no ALS",30/30,7/30,9/10
"GBT, all signals + ALS",30/30,4/30,10/10
"GBT, no ALS",30/30,4/30,10/10
"no training: robust-z sum, all signals + ALS",30/30,16/30,10/10


**Finding (repaired test: held-out insiders scored exactly like normal staff).**

- **Hidden job-hunters (top 100 of 30):** trained logistic regression catches 5 with ALS and 7 without; gradient boosting catches 4. The **label-free robust-z sum, with no attack-label fitting, catches 16**.
- Leakers and sysadmins are caught whatever is hidden.

**Change:** trained combiners' generalization deteriorated sharply for the withheld job-hunter scenario. Their cross-validated scores (up to 60 of 70) are not the headline.

## Step 8: Models from the literature, at day and week granularity

**Why.** Published work on CERT (Le & Zincir-Heywood, TNSM 2021; Tuor et al. 2017) uses autoencoders, Isolation Forest and LOF, and finds that **week-level** data helps for slow attacks. We fitted each on train-window vectors of the person's own log-count deviations (9 counts) and scored test days and weeks; a person's score is the worst period, or the mean of the 3 worst.

In [10]:
F9 = ["n_logon", "n_logon_ah", "n_usb", "n_usb_ah", "n_file", "n_file_ah", "n_http", "n_http_ah", "n_pcs"]

def week_table(c):
    # weekly totals; the week containing the cutoff mixes train and test days, so it is dropped from both
    wk = c.daily.assign(week=c.daily.day.dt.to_period("W").dt.start_time).groupby(["user", "week"])[F9].sum().reset_index()
    wk["part"] = np.where(wk.week + pd.Timedelta(days=6) < c.cut, "train", np.where(wk.week >= c.cut, "test", "boundary"))
    return wk[wk.part != "boundary"].reset_index(drop=True)

def personal(c, df, key):
    v = np.log1p(df[F9].astype(float)); trm = (df.part == "train") if "part" in df else (df[key] < c.cut)
    mu = v[trm].groupby(df.user[trm]).mean(); sd = v[trm].groupby(df.user[trm]).std().fillna(0)
    return ((v - mu.reindex(df.user).values) / (sd.reindex(df.user).values + 0.25)).fillna(0), trm.values

def det_models(c, grans=("day", "week"), which=("IsolationForest", "LOF", "Autoencoder")):
    rows = []
    for gran in grans:
        if gran == "week":
            df = week_table(c); key = "week"
        else:
            df, key = c.daily, "day"
        Z, trm = personal(c, df, key)
        sc_ = StandardScaler().fit(Z.values[trm]); Xtr, Xte = sc_.transform(Z.values[trm]), sc_.transform(Z.values[~trm]); ute = df.user.values[~trm]
        fns = {"IsolationForest": lambda: -IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Xtr).score_samples(Xte),
               "LOF": lambda: -LocalOutlierFactor(n_neighbors=35, novelty=True).fit(Xtr[::3]).score_samples(Xte),
               "Autoencoder": lambda: ((MLPRegressor(hidden_layer_sizes=(6, 3, 6), max_iter=300, random_state=SEED, early_stopping=True)
                                         .fit(Xtr, Xtr).predict(Xte) - Xte) ** 2).sum(axis=1)}
        for name in which:
            g = pd.Series(fns[name](), index=ute).groupby(level=0)
            c.S[f"{gran}_{name}_max"] = g.max(); c.S[f"{gran}_{name}_top3"] = g.apply(lambda x: x.nlargest(3).mean())
            rows += [metrics(f"{gran} {name} (worst)", c.S[f"{gran}_{name}_max"], c),
                     metrics(f"{gran} {name} (mean of 3 worst)", c.S[f"{gran}_{name}_top3"], c)]
    return rows

display(table(det_models(c42)))

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
day IsolationForest (worst),0.85,36.0,0.539,0.921,30/30,2/30,10/10
day IsolationForest (mean of 3 worst),0.80,33.0,0.525,0.913,30/30,2/30,4/10
day LOF (worst),0.65,23.0,0.412,0.893,20/30,3/30,2/10
day LOF (mean of 3 worst),0.60,23.0,0.382,0.895,20/30,3/30,2/10
day Autoencoder (worst),1.00,37.0,0.615,0.885,30/30,0/30,9/10
day Autoencoder (mean of 3 worst),1.00,39.0,0.632,0.893,30/30,0/30,9/10
week IsolationForest (worst),0.10,26.0,0.214,0.724,30/30,2/30,4/10
week IsolationForest (mean of 3 worst),0.15,17.0,0.174,0.661,23/30,1/30,2/10
week LOF (worst),0.45,16.0,0.202,0.636,4/30,12/30,0/10


**Finding.**

- **Day level:** models behave like the baselines (leakers and sysadmins, at most 3 of 30 job-hunters).
- **Week level:** **LOF finds 12 to 13 of 30 job-hunters** in its top 100 (P@20 0.35 to 0.45) while missing the other scenarios. The weekly autoencoder reaches 39 of 70.

**Change:** weekly LOF is the main slow-drift signal.

## Step 9: Recommender tweaks from prior work

**Why.** Research on collaborative filtering for access anomalies (Menon et al. 2014 on hospital records; Google's Facade; Microsoft's Modell et al.; Hu, Koren & Volinsky on implicit ALS) suggested three changes:

- **Peer-of-item:** score a new access by 1 minus the highest cosine similarity between the person's ALS vector and those of the item's previous users ("nobody like you has used this").
- **Lift:** divide predicted affinity by the item's popularity, since implicit ALS favors popular items.
- **28-day site lift:** for every site, the rise in visits over 28 days against the person's own rate, optionally weighted by ALS surprise (aimed at the job-hunters, who already visited job sites before).

In [11]:
def det_als_tweaks(c):
    for pairs_tr, new, item in ((c.pc_tr, c.pc_new.copy(), "pc"), (c.dom_tr, c.dom_new.copy(), "domain")):
        U, V, iidx, cnt = train_als(c, pairs_tr, item)
        Un = U / (np.linalg.norm(U, axis=1, keepdims=True) + 1e-9)
        visitors = cnt.groupby(item).user.apply(lambda s: np.array([c.uidx[u] for u in s]))
        cache = {}
        def peer(u, v):
            if v not in visitors.index: return 1.0
            if (u, v) not in cache: cache[(u, v)] = 1 - float(np.max(Un[visitors[v]] @ Un[c.uidx[u]]))
            return cache[(u, v)]
        new["peer"] = [peer(u, v) for u, v in zip(new.user, new[item])]
        pop = cnt.groupby(item).user.nunique() / len(c.users)
        new["lift"] = np.clip(-np.log(np.clip(predict(c, U, V, iidx, new, item), 1e-4, None) / new[item].map(pop).fillna(1 / len(c.users)).values), 0, None)
        d = pad_days(c, new.groupby(["user", "day"])[["peer", "lift"]].sum().reset_index(), ["peer", "lift"])
        tag = "pc" if item == "pc" else "dom"
        if item == "domain": c.dom_new_peer = new
        c.S[f"{tag}_peer"], c.S[f"{tag}_lift"] = robust_max(d, "peer"), robust_max(d, "lift")

def det_site_lift(c):
    dtr, dte = c.dom[c.dom.day < c.cut], c.dom[c.dom.day >= c.cut]
    U, V, iidx, cnt = train_als(c, dtr, "domain")
    trate = cnt.set_index(["user", "domain"]).c / c.tr.groupby("user").day.nunique().reindex(cnt.user).values
    w = dte.sort_values(["user", "domain", "day"]).set_index("day").groupby(["user", "domain"]).n.rolling("28D").sum().reset_index()
    w["exp"] = trate.reindex(pd.MultiIndex.from_frame(w[["user", "domain"]])).fillna(0).values * 20
    w["lift"] = np.clip((w.n - w.exp) / np.sqrt(w.exp + 1), 0, None)
    w["w"] = w.lift * (1 - np.clip(predict(c, U, V, iidx, w, "domain"), 0, 1))
    agg = w.groupby(["user", "day"]).agg(lift=("lift", "sum"), w=("w", "sum")).reset_index()
    c.S["site_lift"], c.S["site_lift_als"] = agg.groupby("user").lift.max(), agg.groupby("user").w.max()

det_als_tweaks(c42); det_site_lift(c42)
display(table([metrics(f"single: {k}", S[k], c42) for k in ["a_pc", "pc_peer", "pc_lift", "a_dom", "dom_peer", "dom_lift", "site_lift", "site_lift_als"]]))

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
single: a_pc,0.150,10.3,0.099,0.578,4/30,3/30,8/10
single: pc_peer,0.150,6.0,0.104,0.575,4/30,3/30,7/10
single: pc_lift,0.175,11.7,0.097,0.570,3/30,3/30,10/10
single: a_dom,0.000,5.0,0.122,0.713,1/30,8/30,4/10
single: dom_peer,0.450,17.0,0.364,0.839,22.6/30,3/30,9.6/10
single: dom_lift,0.000,13.0,0.157,0.743,0/30,11/30,5/10
single: site_lift,0.050,3.0,0.077,0.553,1/30,0/30,3/10
single: site_lift_als,0.000,1.0,0.088,0.621,1/30,0/30,0/10


**Finding.**

- **Websites:** peer-of-item reaches AUC 0.839 (plain ALS 0.713), but only P@20 0.45 and 17 of 70 once zero-novelty days are counted. The earlier "39 of 70" came from the uncorrected normalization.
- **Computers:** no recommender variant beats the naive check.
- **Lift correction:** did not help.
- **28-day site lift:** this simple version failed (AUC 0.55 to 0.62). Its rolling-window bookkeeping was simplistic, so it isn't a fair test of the idea.

**Change:** peer-of-item stays only as a candidate website signal.

## Step 10: Own-history percentiles (the literature's top tip)

**Why.** Le & Zincir-Heywood (TNSM 2021) found that expressing each period as a **percentile of the same person's own history** beat z-score features and transferred across CERT releases. Weekly totals; for each test week and count, the share of the person's train weeks below it. Four person-level scores:

- `pw_max`: the most unusual week overall (sum of percentiles)
- `pw_hi_max`: the most counts in one week that exceed everything the person did in training (99th percentile and up)
- `pw_hi_weeks`: **how many weeks** had 2 or more such counts (sustained drift)
- `pw_top4`: mean of the 4 most unusual weeks

In [12]:
def det_ownpct(c):
    wk = week_table(c)
    tr, te = wk[wk.part == "train"], wk[wk.part == "test"].copy()
    ref = {u: g[F9].values for u, g in tr.groupby("user")}
    P = np.zeros((len(te), len(F9)))
    for i, (u, row) in enumerate(zip(te.user.values, te[F9].values)):
        r = ref.get(u)
        if r is not None and len(r): P[i] = ((r < row).sum(axis=0) + 0.5 * (r == row).sum(axis=0)) / len(r)
    te["pct_sum"], te["pct_hi"] = P.sum(axis=1), (P >= 0.99).sum(axis=1)
    g = te.groupby("user")
    c.S["pw_max"], c.S["pw_hi_max"] = g.pct_sum.max(), g.pct_hi.max()
    c.S["pw_hi_weeks"] = te.assign(f=te.pct_hi >= 2).groupby("user").f.sum()
    c.S["pw_top4"] = g.pct_sum.apply(lambda x: x.nlargest(4).mean())

det_ownpct(c42)
display(table([metrics(f"single: {k}", S[k], c42) for k in ["pw_max", "pw_hi_max", "pw_hi_weeks", "pw_top4"]]))

BASE = L6 + ["week_LOF_top3", "week_Autoencoder_top3"]
PW = ["pw_max", "pw_hi_max", "pw_hi_weeks", "pw_top4"]
SETS = {"behavior + peers + windows + IF + weekly LOF/AE (no ALS)": BASE,
        "+ ALS novelty": BASE + ALS3,
        "+ own-history percentiles": BASE + PW,
        "+ own-history percentiles + ALS": BASE + PW + ALS3,
        "+ own-history percentiles + ALS peer-of-item": BASE + PW + ["pc_peer", "dom_peer"]}
rows, lrows = [], []
for k, cols in SETS.items():
    rows.append(metrics("LR: " + k, cv_score(c42, cols), c42)); lrows.append({"combiner": "LR: " + k, **loso(c42, cols)})
display(table(rows)); display(pd.DataFrame(lrows).set_index("combiner"))

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
single: pw_max,1.00,37.0,0.603,0.901,30/30,2/30,7/10
single: pw_hi_max,1.00,39.4,0.666,0.947,30/30,4.9/30,8.1/10
single: pw_hi_weeks,0.20,24.3,0.325,0.913,22.4/30,12.5/30,1.1/10
single: pw_top4,0.45,24.0,0.310,0.864,26/30,2/30,6/10


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
LR: behavior + peers + windows + IF + weekly LOF/AE (no ALS),1.0,47.0,0.800,0.973,30/30,17/30,9/10
LR: + ALS novelty,1.0,51.0,0.831,0.978,30/30,20/30,10/10
LR: + own-history percentiles,1.0,51.0,0.836,0.980,30/30,21/30,9/10
LR: + own-history percentiles + ALS,1.0,51.0,0.857,0.984,30/30,23/30,9/10
LR: + own-history percentiles + ALS peer-of-item,1.0,51.0,0.846,0.982,30/30,23/30,10/10


,S1 hidden,S2 hidden,S3 hidden
combiner,,,
LR: behavior + peers + windows + IF + weekly LOF/AE (no ALS),30/30,8/30,10/10
LR: + ALS novelty,30/30,9/30,10/10
LR: + own-history percentiles,30/30,12/30,10/10
LR: + own-history percentiles + ALS,30/30,12/30,10/10
LR: + own-history percentiles + ALS peer-of-item,30/30,16/30,10/10


**Finding.**

- **Single signals:** `pw_hi_max` reaches P@20 1.00 and 39.4 of 70, with AP 0.666 (R2's AP of 0.728 is higher on r4.2). `pw_hi_weeks` catches 12.5 of 30 job-hunters.
- **Learned combiner:** base 47; plus own-history percentiles, 51; plus ALS too, still 51 (AP 0.857).
- **Hidden job-hunters:** 8, then 12 with percentiles, then 16 with website peer-of-item added.

**Change:** own-history percentiles are a core detector.

## Step 11: Combining without labels

**Why.** A trained combiner needs past incidents, which most organizations don't have, and it generalizes poorly (Step 7). We tried label-free fusion from two sources, with all parameters fixed in advance:

- **Industry practice** (Microsoft Sentinel and Purview, Splunk, Exabeam, Google SecOps): group detectors into families, take the strongest signal per family, then combine; Google's rule is strongest family + 0.2 × the rest; Purview spotlights only alerts **corroborated** by several conditions.
- **Statistics for combining evidence:** turn each detector's rank into an empirical p-value (rank / staff), then Fisher, Stouffer, Cauchy combination (ACAT, robust to correlated tests), harmonic mean p, voting, and a trimmed mean of ranks. "Family-first" takes the best p in each family (times family size, a Bonferroni correction) before combining across families.

In [13]:
FAM = {"behavior": ["r1", "r1_top3", "r_w7", "r2", "r_w56"],
       "own_pct": ["pw_max", "pw_hi_max", "pw_hi_weeks", "pw_top4"],
       "peer": ["peer_daily", "peer_w28"],
       "rare": ["w_rare"],
       "drift": ["week_LOF_top3"],
       "recommender": ["a_pc", "a_dom", "dom_peer", "pc_peer"],
       "global": ["iforest", "week_Autoencoder_top3"]}

def fusion_rows(c, fam=FAM, store=True):
    cols = [k for f in fam.values() for k in f]
    X = pd.DataFrame({k: pd.Series(c.S[k]).reindex(c.users) for k in cols}).fillna(0); n = len(X)
    P = X.rank(ascending=False, method="max") / n                    # empirical p-value, 1/n = most extreme person
    tail = -np.log10(P + 1 / n)                                       # empirical tail score
    fam_tail = pd.DataFrame({f: tail[k].max(axis=1) for f, k in fam.items()})
    srt = np.sort(fam_tail.values, axis=1)[:, ::-1]
    hot = ((fam_tail.rank(pct=True) >= 0.99).sum(axis=1) >= 2)
    rows = [metrics("industry: average of family maxima", fam_tail.mean(axis=1), c),
            metrics("industry: top family + 0.2 x rest", srt[:, 0] + 0.2 * srt[:, 1:].sum(axis=1), c),
            metrics("industry: corroborated (2+ families >99th pct) first", srt[:, 0] + 0.2 * srt[:, 1:].sum(axis=1) + 100 * hot.values, c)]
    def combos(Pm, tag):
        k = Pm.shape[1]; Pc = Pm.clip(upper=1 - 1e-9)
        return [metrics(f"{tag} Fisher", (-2 * np.log(Pm)).sum(axis=1), c),
                metrics(f"{tag} Stouffer", norm.isf(Pc).sum(axis=1) / np.sqrt(k), c),
                metrics(f"{tag} Cauchy (ACAT)", np.tan((0.5 - Pc) * np.pi).mean(axis=1), c),
                metrics(f"{tag} harmonic mean p", (1 / Pm).mean(axis=1), c),
                metrics(f"{tag} vote (p <= 0.05)", (Pm <= 0.05).sum(axis=1) + 1e-3 * (1 / Pm).mean(axis=1), c),
                metrics(f"{tag} trimmed mean of ranks", np.sort(1 - Pm, axis=1)[:, 1:-1].mean(axis=1), c)]
    rows += combos(P, "all detectors:")
    Pf = pd.DataFrame({f: (P[k].min(axis=1) * len(k)).clip(upper=1) for f, k in fam.items()})
    rows += combos(Pf, "family-first:")
    if store:   # keep the frozen family-first Stouffer (no ownership family) for Step 14's equal-budget table
        c.fused_stouffer = pd.Series(norm.isf(Pf.clip(upper=1 - 1e-9).values).sum(axis=1) / np.sqrt(Pf.shape[1]), index=c.users)
    gate = (hot, int(hot.sum()), int(sum(u in c.lab for u in hot[hot].index)))
    return rows, gate

rows, gate = fusion_rows(c42)
rows.append(metrics("reference: best single (R2 28-day window)", S["r2"], c42))
display(table(rows))
print(f"people corroborated by 2+ families above their 99th percentile: {gate[1]}, of whom insiders: {gate[2]}")

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
industry: average of family maxima,1.00,43.0,0.745,0.952,30/30,7/30,10/10
industry: top family + 0.2 x rest,0.95,44.0,0.680,0.925,30/30,7/30,10/10
industry: corroborated (2+ families >99th pct) first,0.95,44.0,0.680,0.925,30/30,7/30,10/10
all detectors: Fisher,1.00,42.0,0.722,0.953,30/30,5/30,10/10
all detectors: Stouffer,0.90,39.0,0.660,0.945,28/30,3/30,10/10
all detectors: Cauchy (ACAT),0.75,25.0,0.493,0.923,27/30,3/30,10/10
all detectors: harmonic mean p,0.75,42.0,0.614,0.926,30/30,6/30,10/10
all detectors: vote (p <= 0.05),0.85,40.0,0.610,0.912,30/30,3/30,10/10
all detectors: trimmed mean of ranks,0.95,41.0,0.719,0.959,30/30,10/30,10/10


people corroborated by 2+ families above their 99th percentile: 11, of whom insiders: 11


**Finding.** Label-free fusion on r4.2:

| Method | P@20 | hits@70 | AP |
|---|---|---|---|
| Family-first Stouffer | 1.00 | 45 | 0.784 |
| Family-first Cauchy | 1.00 | 46 | 0.787 |
| Best single signal (R2) | 1.00 | 38 | 0.728 |

- Treating every detector separately is worse, because correlated detectors outvote the rest.
- **Corroboration:** all 11 people flagged by 2 or more families above their 99th percentile were insiders.

**Change:** family-first Stouffer was frozen as the label-free method to test on r5.2.

## Step 12: Generalization test on r5.2 (the pipeline frozen, run unchanged)

**Why.** Everything so far was developed on r4.2 with its answer key in view, which risks fitting to it. r5.2 is a different simulation (2,000 staff, 99 insiders) and adds **scenario 4, the snooper who logs into colleagues' computers**, exactly the "access that doesn't fit your role" case the recommender was designed for. Nothing below was tuned on r5.2. The train window ends April 1, 2010 because scenario 4 starts April 16.

In [14]:
def build_all(c):
    t = time.time()
    det_baselines(c); det_novelty(c); det_more(c)
    det_models(c, grans=("week",), which=("LOF", "Autoencoder"))
    det_als_tweaks(c); det_site_lift(c); det_ownpct(c)
    print(f"r{c.rel}: all detectors built in {time.time() - t:.0f}s")

c52 = context("5.2"); build_all(c52)
keys = ["r0", "r1", "r2", "n_pc", "a_pc", "pc_peer", "n_dom", "a_dom", "dom_peer", "w_rare",
        "peer_daily", "iforest", "week_LOF_top3", "pw_max", "pw_hi_max", "pw_hi_weeks", "pw_top4"]
display(table([metrics(f"r5.2 single: {k}", c52.S[k], c52) for k in keys]))

r5.2: loaded in 1s | staff 2000 | insiders 99 | days 517 | train days 89 | test days 428


r5.2: all detectors built in 62s


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100,S4 top100
detector,,,,,,,,
r5.2 single: r0,0.050,4.0,0.114,0.694,4/29,4.6/30,1/10,0/30
r5.2 single: r1,0.750,32.0,0.358,0.811,25/29,2/30,9/10,2/30
r5.2 single: r2,0.900,36.0,0.428,0.783,25/29,5/30,7/10,2/30
r5.2 single: n_pc,0.258,13.2,0.112,0.683,1.8/29,2.4/30,3/10,10.9/30
r5.2 single: a_pc,0.100,11.0,0.119,0.681,0.2/29,3/30,10/10,5.3/30
r5.2 single: pc_peer,0.250,13.0,0.124,0.687,1/29,0/30,3/10,13/30
r5.2 single: n_dom,0.100,3.8,0.072,0.589,0/29,2.5/30,2/10,0.5/30
r5.2 single: a_dom,0.050,4.0,0.073,0.626,0/29,5/30,1/10,1/30
r5.2 single: dom_peer,0.350,31.0,0.259,0.720,29/29,0/30,10/10,0/30


In [15]:
rows, lrows = [], []
for k, cols in SETS.items():
    rows.append(metrics("r5.2 LR: " + k, cv_score(c52, cols), c52)); lrows.append({"combiner": "r5.2 LR: " + k, **loso(c52, cols)})
display(table(rows)); display(pd.DataFrame(lrows).set_index("combiner"))
rows52, gate52 = fusion_rows(c52)
rows52.append(metrics("r5.2 reference: R2 28-day window", c52.S["r2"], c52))
display(table(rows52))
print(f"r5.2 people corroborated by 2+ families: {gate52[1]}, of whom insiders: {gate52[2]}")

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100,S4 top100
detector,,,,,,,,
r5.2 LR: behavior + peers + windows + IF + weekly LOF/AE (no ALS),0.90,35.0,0.471,0.908,28/29,6/30,4/10,4/30
r5.2 LR: + ALS novelty,0.65,32.0,0.441,0.936,21/29,6/30,9/10,10/30
r5.2 LR: + own-history percentiles,0.80,44.0,0.548,0.944,27/29,11/30,6/10,9/30
r5.2 LR: + own-history percentiles + ALS,0.65,41.0,0.545,0.956,26/29,11/30,9/10,10/30
r5.2 LR: + own-history percentiles + ALS peer-of-item,0.70,44.0,0.567,0.957,28/29,7/30,10/10,10/30


,S1 hidden,S2 hidden,S3 hidden,S4 hidden
combiner,,,,
r5.2 LR: behavior + peers + windows + IF + weekly LOF/AE (no ALS),28/29,0/30,4/10,3/30
r5.2 LR: + ALS novelty,0/29,0/30,9/10,5/30
r5.2 LR: + own-history percentiles,24/29,3/30,6/10,2/30
r5.2 LR: + own-history percentiles + ALS,1/29,1/30,10/10,3/30
r5.2 LR: + own-history percentiles + ALS peer-of-item,6/29,0/30,10/10,1/30


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100,S4 top100
detector,,,,,,,,
industry: average of family maxima,0.80,38.0,0.468,0.911,27/29,5/30,10/10,2/30
industry: top family + 0.2 x rest,0.65,32.0,0.419,0.906,25/29,3/30,10/10,2/30
industry: corroborated (2+ families >99th pct) first,0.70,32.0,0.418,0.906,25/29,3/30,10/10,2/30
all detectors: Fisher,0.85,38.0,0.458,0.888,26/29,4/30,10/10,2/30
all detectors: Stouffer,0.70,29.0,0.354,0.869,21/29,2/30,10/10,3/30
all detectors: Cauchy (ACAT),0.60,18.0,0.262,0.849,5/29,2/30,10/10,3/30
all detectors: harmonic mean p,0.60,29.0,0.365,0.896,25/29,2/30,10/10,2/30
all detectors: vote (p <= 0.05),0.70,36.0,0.412,0.892,25/29,4/30,10/10,2/30
all detectors: trimmed mean of ranks,0.75,32.0,0.363,0.875,22/29,5/30,10/10,2/30


r5.2 people corroborated by 2+ families: 20, of whom insiders: 14


**Finding.** r5.2 is harder.

- **Own-history percentile:** `pw_hi_max` reaches **P@20 1.00, 40.3 of 99, AP 0.477** with all 29 leakers, the best single signal on r5.2. R2 reaches 36 and AP 0.428.
- **Frozen family-first Stouffer:** P@20 0.90, 38, AP 0.478. That beats R2 and ties the percentile.
- **Snoopers, daily aggregation:** the recommender's computer signals catch 5.3 (ALS) and 13 (peer-of-item) of 30 in the top 100, against 10.9 for naive. Step 13 shows the aggregation matters more than the scoring idea.
- **Learned combiners:** cross-validation reaches up to 49 of 99 (Step 15). With one scenario hidden, results swing widely, with hidden leakers ranging from 0 to 28 of 29.
- **Corroboration:** 20 people, 14 of them insiders (70%).

## Step 13: An ownership rule for personal computers

**Why.** Step 12 showed collaborative filtering forgiving the snooper: the colleague's computer belongs to someone similar, so the model rates the access as normal. General security logic says a **personal** resource is different from a **shared** one: if a computer has one main user, anyone else logging in is out of role however similar they are.

- A computer is **personal** if one person accounts for 80% or more of its train-window login-days.
- `own_days`: test days with any login to someone else's personal computer.
- `own_weeks_adj`: test **weeks** with such a login, divided by (1 + 50 × the person's train-window rate of such logins), so people whose job already involves roaming (e.g. admins) are discounted.

The rule uses no labels. It was designed after seeing Step 12's failure, so it has not had an independent test of its own (see open questions).

In [16]:
def det_owner(c):
    tr, te = c.pcs[c.pcs.day < c.cut], c.pcs[c.pcs.day >= c.cut].copy()
    days = tr.groupby(["pc", "user"]).size().rename("d").reset_index(); tot = days.groupby("pc").d.sum()
    top = days.sort_values("d", ascending=False).drop_duplicates("pc").set_index("pc")
    owner = top.user[(top.d / tot.reindex(top.index)) >= 0.8]
    te["foreign"] = (te.pc.map(owner).notna() & (te.pc.map(owner) != te.user)).astype(int)
    d = te.groupby(["user", "day"]).foreign.sum().reset_index()
    wk = d.assign(week=d.day.dt.to_period("W").dt.start_time).groupby(["user", "week"]).foreign.sum().reset_index()
    base = tr.assign(f=(tr.pc.map(owner).notna() & (tr.pc.map(owner) != tr.user)).astype(int)).groupby("user").f.mean()
    c.S["own_days"] = d.groupby("user").foreign.apply(lambda x: (x > 0).sum()).reindex(c.users).fillna(0)
    weeks = wk.groupby("user").foreign.apply(lambda x: (x > 0).sum()).reindex(c.users).fillna(0)
    c.S["own_weeks_adj"] = weeks / (1 + 50 * base.reindex(c.users).fillna(0))
    return len(owner), c.pcs.pc.nunique()

for c in (c42, c52):
    n_own, n_pc = det_owner(c)
    print(f"r{c.rel}: {n_own} of {n_pc} computers are personal")
    display(table([metrics(f"r{c.rel} ownership: own_days", c.S["own_days"], c),
                   metrics(f"r{c.rel} ownership: own_weeks_adj", c.S["own_weeks_adj"], c),
                   metrics(f"r{c.rel} reference: naive new computers", c.S["n_pc"], c),
                   metrics(f"r{c.rel} reference: ALS surprising computers", c.S["a_pc"], c)]))

def det_owner_ablation(c):
    # control: give the naive and recommender computer signals the SAME weekly counting and roaming discount
    tr, te = c.pcs[c.pcs.day < c.cut], c.pcs[c.pcs.day >= c.cut].copy()
    seen = set(zip(tr.user, tr.pc))
    primary = tr.groupby(["user", "pc"]).size().rename("d").reset_index().sort_values("d", ascending=False).drop_duplicates("user").set_index("user").pc
    base_away = tr.assign(f=(tr.pc != tr.user.map(primary)).astype(int)).groupby("user").f.mean()   # train rate of logins away from one's main PC
    te["new"] = np.array([(u, q) not in seen for u, q in zip(te.user, te.pc)])
    U, V, iidx, _ = train_als(c, tr, "pc")
    te["surprise"] = 1 - np.clip(predict(c, U, V, iidx, te, "pc"), 0, 1)
    def weekly(flag, base):
        d = te.assign(f=flag.astype(int)).groupby(["user", "day"]).f.sum().reset_index()
        wk = d.assign(week=d.day.dt.to_period("W").dt.start_time).groupby(["user", "week"]).f.sum()
        return (wk > 0).groupby(level=0).sum().reindex(c.users).fillna(0) / (1 + 50 * base.reindex(c.users).fillna(0))
    days = tr.groupby(["pc", "user"]).size().rename("d").reset_index(); tot = days.groupby("pc").d.sum()
    topu = days.sort_values("d", ascending=False).drop_duplicates("pc").set_index("pc")
    owner = topu.user[(topu.d / tot.reindex(topu.index)) >= 0.8]
    te["foreign"] = te.pc.map(owner).notna().values & (te.pc.map(owner) != te.user).values
    c.S["abl_own_weeks_common"] = weekly(te.foreign, base_away)          # same discount for all three
    c.S["abl_new_weeks_adj"] = weekly(te.new, base_away)
    c.S["abl_als_weeks_adj"] = weekly(te.new & (te.surprise >= 0.5), base_away)

for c in (c42, c52):
    det_owner_ablation(c)
    display(table([metrics(f"r{c.rel} same weekly+discount: ownership (someone else's personal PC)", c.S["abl_own_weeks_common"], c),
                   metrics(f"r{c.rel} same weekly+discount: naive (any computer new to you)", c.S["abl_new_weeks_adj"], c),
                   metrics(f"r{c.rel} same weekly+discount: recommender (new and surprising, >= 0.5)", c.S["abl_als_weeks_adj"], c)]))

FAM_OWN = {**FAM, "ownership": ["own_weeks_adj", "own_days"]}
for c in (c42, c52):
    r_, _ = fusion_rows(c, FAM_OWN, store=False)
    display(table([r for r in r_ if "family-first" in r["detector"]][:3]))

r4.2: 999 of 1003 computers are personal


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
r4.2 ownership: own_days,0.000,11.0,0.095,0.578,4.1/30,3.1/30,10/10
r4.2 ownership: own_weeks_adj,0.000,2.0,0.078,0.570,4.1/30,3.1/30,10/10
r4.2 reference: naive new computers,0.231,11.7,0.101,0.579,4.6/30,2.4/30,7/10
r4.2 reference: ALS surprising computers,0.150,10.3,0.099,0.578,4/30,3/30,8/10


r5.2: 1984 of 2004 computers are personal


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100,S4 top100
detector,,,,,,,,
r5.2 ownership: own_days,0.000,7.0,0.101,0.674,0/29,2/30,10/10,2/30
r5.2 ownership: own_weeks_adj,0.800,30.0,0.272,0.688,1/29,1/30,0/10,30/30
r5.2 reference: naive new computers,0.258,13.2,0.112,0.683,1.8/29,2.4/30,3/10,10.9/30
r5.2 reference: ALS surprising computers,0.100,11.0,0.119,0.681,0.2/29,3/30,10/10,5.3/30


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
r4.2 same weekly+discount: ownership (someone else's personal PC),0.0,3.0,0.078,0.570,4.1/30,3.1/30,10/10
r4.2 same weekly+discount: naive (any computer new to you),0.0,3.0,0.078,0.569,3/30,3/30,6/10
"r4.2 same weekly+discount: recommender (new and surprising, >= 0.5)",0.0,3.0,0.077,0.567,4/30,3/30,9/10


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100,S4 top100
detector,,,,,,,,
r5.2 same weekly+discount: ownership (someone else's personal PC),0.85,24.0,0.262,0.686,0/29,1/30,0/10,27/30
r5.2 same weekly+discount: naive (any computer new to you),0.70,24.0,0.225,0.691,1/29,1/30,0/10,27/30
"r5.2 same weekly+discount: recommender (new and surprising, >= 0.5)",0.75,23.0,0.230,0.687,0/29,1/30,0/10,27/30


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
family-first: Fisher,0.90,42.0,0.672,0.934,30/30,5/30,10/10
family-first: Stouffer,0.95,40.0,0.698,0.952,30/30,4/30,10/10
family-first: Cauchy (ACAT),0.90,40.0,0.679,0.952,30/30,6/30,10/10


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100,S4 top100
detector,,,,,,,,
family-first: Fisher,0.6,37.0,0.440,0.930,26/29,5/30,10/10,5/30
family-first: Stouffer,0.6,35.0,0.394,0.918,23/29,4/30,10/10,4/30
family-first: Cauchy (ACAT),0.5,32.0,0.348,0.920,22/29,2/30,10/10,8/30


**Finding.**

- **The ownership rule:** `own_weeks_adj` puts **all 30 snoopers in the top 100** (P@20 0.80), while `own_days` puts only 2. The weekly counting matters enormously.
- **Controlled comparison:** we gave ownership, naive "computer new to you", and recommender-flagged signals the *same* weekly counting and the *same* roaming discount. Each catches **27 of 30** snoopers (P@20 0.85, 0.70, 0.75).
- **What actually catches snoopers:** persistence, meaning logging into unusual computers week after week, discounted for people who already roamed in training. The ownership idea adds a little top-of-list precision.
- **Correction:** the earlier explanation, that collaborative filtering forgives a colleague's computer, is **not supported** once aggregation is controlled.
- Folded into fusion, the ownership family keeps only 4 to 8 of 30 snoopers.

**Change:** the out-of-role signal is persistence-based. It was designed after seeing r5.2 and needs an independent test.

## Step 14: Specialist alert queues instead of one fused score

**Why.** Steps 11 and 13 show averaging washing out specialist detectors. Commercial tools give each detection type its own alerts. Four queues, chosen in advance, one signal each:

| Queue | Signal | Catches |
|---|---|---|
| Sudden spike | `pw_hi_max` (own-history percentile) | leakers, sysadmins |
| Slow drift | `pw_hi_weeks` (unusual weeks) | slow ramps |
| Out-of-role access | `own_weeks_adj` (ownership rule) | snoopers |
| Unusual destination | `dom_peer` (ALS peer-of-item on websites) | rare or out-of-profile sites |

Two ways to size them: **(a)** 25 review slots each (100 total), and **(b)** a preset threshold, alerting only when a person's robust z-score on that signal is 6 or more (far outside normal), so an absent threat produces few alerts.

In [17]:
QUEUES = {"sudden spike": "pw_hi_max", "slow drift": "pw_hi_weeks", "out-of-role access": "own_weeks_adj", "unusual destination": "dom_peer"}
SECOND = {"pw_hi_max": "pw_max", "pw_hi_weeks": "pw_max", "own_weeks_adj": "own_days", "dom_peer": "w_rare",
          "w_rare": "n_dom", "n_dom": "w_rare", "week_LOF_top3": "pw_max"}   # tie-breakers chosen a priori (a related signal)

def ranked(c, sig):
    df = pd.DataFrame({"a": aligned(c.S[sig], c), "b": aligned(c.S[SECOND.get(sig, sig)], c)})
    return list(df.sort_values(["a", "b"], ascending=False, kind="mergesort").index)

def queues(c, mode="slots", k=25, z=6.0, qmap=None):
    rows, picked = [], set()
    for q, sig in (qmap or QUEUES).items():
        s = aligned(c.S[sig], c)
        sel = ranked(c, sig)[:k] if mode == "slots" else list(s.index[rz(s, c.users) >= z])
        picked |= set(sel)
        row = {"queue": q, "signal": sig, "alerts": len(sel), "insiders": sum(u in c.lab for u in sel)}
        row.update({f"S{sc}": sum(1 for u in sel if c.lab.get(u) == sc) for sc in sorted(set(c.lab.values()))})
        rows.append(row)
    tot = {sc: sum(1 for v in c.lab.values() if v == sc) for sc in sorted(set(c.lab.values()))}
    rows.append({"queue": "ALL QUEUES (unique people)", "signal": "", "alerts": len(picked), "insiders": sum(u in c.lab for u in picked),
                 **{f"S{sc}": f"{sum(1 for u in picked if c.lab.get(u) == sc)}/{tot[sc]}" for sc in tot}})
    return pd.DataFrame(rows).set_index("queue")

def queue_budget(c, K=100, qmap=None):
    # round-robin across queues, skipping people already picked, until exactly K different people are reviewed
    assert K <= len(c.users), 'budget larger than staff'
    qmap = qmap or QUEUES; names = list(qmap); lists = [ranked(c, s) for s in qmap.values()]
    ptr, picked, src = [0] * len(lists), [], {}
    while len(picked) < K:
        for qi, L in enumerate(lists):
            while ptr[qi] < len(L) and L[ptr[qi]] in src: ptr[qi] += 1
            if ptr[qi] < len(L) and len(picked) < K:
                u = L[ptr[qi]]; picked.append(u); src[u] = names[qi]; ptr[qi] += 1
    return picked, src

def budget_people(name, people, c):
    row = {"method": name, "people": len(people), "insiders": sum(u in c.lab for u in people)}
    row.update({f"S{sc}": f"{sum(1 for u in people if c.lab.get(u) == sc)}/{sum(1 for v in c.lab.values() if v == sc)}" for sc in sorted(set(c.lab.values()))})
    return row

def budget_score(name, s, c, K=100):
    v = aligned(s, c).values.astype(float); y = np.array([u in c.lab for u in c.users])
    row = {"method": name, "people": K, "insiders": round(top_expected(v, y, K), 1)}
    for sc in sorted(set(c.lab.values())):
        m = np.array([c.lab.get(u) == sc for u in c.users]); row[f"S{sc}"] = fmt(top_expected(v, m, K), int(m.sum()))
    return row

BEST_LR = BASE + PW + ["pc_peer", "dom_peer"]
for c in (c42, c52):
    print(f"r{c.rel}, (a) 25 slots per queue (ties broken by a related signal)"); display(queues(c, "slots"))
    print(f"r{c.rel}, (b) threshold: robust z >= 6"); display(queues(c, "threshold"))
    people, _ = queue_budget(c, 100)
    print(f"r{c.rel}, (c) equal budget: exactly 100 different people reviewed by every method")
    display(pd.DataFrame([budget_people("four queues, round-robin refill", people, c),
                          budget_score("baseline: R2 28-day window", c.S["r2"], c),
                          budget_score("best single: own-history percentile", c.S["pw_hi_max"], c),
                          budget_score("label-free fusion: family-first Stouffer", c.fused_stouffer, c),
                          budget_score("trained combiner (needs labels; CV)", cv_score(c, BEST_LR), c)]).set_index("method"))

r4.2, (a) 25 slots per queue (ties broken by a related signal)


,signal,alerts,insiders,S1,S2,S3
queue,,,,,,
sudden spike,pw_hi_max,25,25,25,0,0
slow drift,pw_hi_weeks,25,9,5,4,0
out-of-role access,own_weeks_adj,25,0,0,0,0
unusual destination,dom_peer,25,9,0,0,9
ALL QUEUES (unique people),,92,38,25/30,4/30,9/10


r4.2, (b) threshold: robust z >= 6


,signal,alerts,insiders,S1,S2,S3
queue,,,,,,
sudden spike,pw_hi_max,0,0,0,0,0
slow drift,pw_hi_weeks,0,0,0,0,0
out-of-role access,own_weeks_adj,1,0,0,0,0
unusual destination,dom_peer,9,8,0,0,8
ALL QUEUES (unique people),,10,8,0/30,0/30,8/10


r4.2, (c) equal budget: exactly 100 different people reviewed by every method


...

,signal,alerts,insiders,S1,S2,S3,S4
queue,,,,,,,
sudden spike,pw_hi_max,25,24,24,0,0,0
slow drift,pw_hi_weeks,25,5,0,2,1,2
out-of-role access,own_weeks_adj,25,20,0,0,0,20
unusual destination,dom_peer,25,8,2,0,6,0
ALL QUEUES (unique people),,94,53,24/29,2/30,7/10,20/30


r5.2, (b) threshold: robust z >= 6


,signal,alerts,insiders,S1,S2,S3,S4
queue,,,,,,,
sudden spike,pw_hi_max,0,0,0,0,0,0
slow drift,pw_hi_weeks,0,0,0,0,0,0
out-of-role access,own_weeks_adj,0,0,0,0,0,0
unusual destination,dom_peer,7,6,0,0,6,0
ALL QUEUES (unique people),,7,6,0/29,0/30,6/10,0/30


r5.2, (c) equal budget: exactly 100 different people reviewed by every method


,people,insiders,S1,S2,S3,S4
method,,,,,,
"four queues, round-robin refill",100,53.0,24/29,2/30,7/10,20/30
baseline: R2 28-day window,100,39.0,25/29,5/30,7/10,2/30
best single: own-history percentile,100,44.6,29/29,5.4/30,9/10,1.3/30
label-free fusion: family-first Stouffer,100,43.0,29/29,4/30,10/10,0/30
trained combiner (needs labels; CV),100,55.0,28/29,7/30,10/10,10/30


**Finding.**

- **Fixed slots (25 per queue, ties broken by a related signal):** 38 insiders among 92 people (r4.2), 53 among 94 (r5.2).
- **Equal budget, exactly 100 different people:**

| Method | r4.2 | r5.2 |
|---|---|---|
| Four queues | 42 | **53** |
| Baseline R2 | 44 | 39 |
| Own-history percentile | 43 | 44.6 |
| Frozen family-first Stouffer | **55** | 43 |
| Trained combiner (needs labels) | 63 | 55 |

- **Preset threshold** (robust z of 6 or more): far too strict, with 10 and 7 alerts.

**Reading:** queues win only when an attack type needs its own specialist (r5.2's snoopers). Without one, fusion wins clearly (r4.2). Neither design wins on both releases.

## Step 15: How much does the recommender actually contribute?

**Why.** Two questions people ask: does the recommender learn anything real, and how much does it add to catching insiders? RMSE (root mean squared error) only fits the first question, because insider detection is a ranking task with no number to predict. So we measure three things:

1. **The recommender's own job:** predicting which sites (and computers) a *normal* employee uses in the test period, versus a "just guess the popular ones" baseline. We report RMSE on the visit/no-visit grid and, because implicit-feedback recommenders are judged by ranking (Hu, Koren & Volinsky), the **mean percentile rank** of sites a person uses for the first time (0% = it was the model's top guess, 50% = random).
2. **Website signals on identical footing:** naive "new site", "rare new site", recommender surprise and recommender peer-of-item, all counted per active day with the same normalization, so only the scoring idea differs.
3. **Ablations:** the trained combiner with no website novelty, with naive novelty, with the recommender, with both (cross-validation and the unseen-attack test); and the four queues at a fixed budget of 100 people with the destination queue powered by the recommender, by rare sites, by naive novelty, or removed.

In [18]:
def als_accuracy(c, pairs, item):
    tr, te = pairs[pairs.day < c.cut], pairs[pairs.day >= c.cut]
    normal = [u for u in c.users if u not in c.lab]; pos = {u: i for i, u in enumerate(normal)}
    U, V, iidx, cnt = train_als(c, tr, item)
    order_items = sorted(iidx, key=iidx.get)
    pop = cnt.groupby(item).user.nunique().reindex(order_items).fillna(0).values / len(c.users)
    P = np.clip(U[[c.uidx[u] for u in normal]] @ V.T, 0, 1); B = np.tile(pop, (len(normal), 1))
    Y = np.zeros_like(P); Trn = np.zeros_like(P)
    tep = te[te.user.isin(pos) & te[item].isin(iidx)][["user", item]].drop_duplicates()
    Y[tep.user.map(pos).values, tep[item].map(iidx).values] = 1
    trp = cnt[cnt.user.isin(pos)]; Trn[trp.user.map(pos).values, trp[item].map(iidx).values] = 1
    newm = Trn == 0
    rmse = lambda A, M: float(np.sqrt((((A - Y) ** 2)[M]).mean()))
    def mpr(A):
        ranks = []
        for i in range(len(normal)):
            cand = np.where(newm[i])[0]
            if len(cand) < 2 or Y[i, cand].sum() == 0: continue
            r = (rankdata(-A[i, cand], method="average") - 1) / (len(cand) - 1)
            ranks.extend(r[Y[i, cand] == 1])
        return 100 * float(np.mean(ranks)), len(ranks)
    m_als, n_new = mpr(U[[c.uidx[u] for u in normal]] @ V.T); m_pop, _ = mpr(B)   # rank unclipped affinities
    return {"release": c.rel, "resource": "websites" if item == "domain" else "computers", "items": len(iidx), "first-time uses scored": n_new,
            "RMSE all pairs: ALS": round(rmse(P, np.ones_like(P, bool)), 4), "RMSE all pairs: popularity": round(rmse(B, np.ones_like(P, bool)), 4),
            "RMSE unseen pairs: ALS": round(rmse(P, newm), 4), "RMSE unseen pairs: popularity": round(rmse(B, newm), 4),
            "MPR first-time uses: ALS %": round(m_als, 1), "MPR first-time uses: popularity %": round(m_pop, 1), "MPR random %": 50.0}

display(pd.DataFrame([als_accuracy(c, pr, it) for c in (c42, c52) for pr, it in ((c.dom, "domain"), (c.pcs, "pc"))]).set_index(["release", "resource"]))

items  first-time uses scored  RMSE all pairs: ALS  RMSE all pairs: popularity  RMSE unseen pairs: ALS  RMSE unseen pairs: popularity  MPR first-time uses: ALS %  MPR first-time uses: popularity %  \
release resource                                                                                                                                                                                                          
4.2     websites     461                   24605               0.5394                      0.4257                  0.6066                         0.3206                        35.9                               35.5   
        computers   1002                    9943               0.1392                      0.1244                  0.1268                         0.1035                        48.1                               46.0   
5.2     websites     462                   81689               0.5191                      0.4426                  0.5807                         0.3464                        35.8                               38.3   
        computers   2004                   42339               0.1173                      0.1131                  0.1110                         0.1054                        49.0                               47.5   

                   MPR random %  
release resource                 
4.2     websites           50.0  
        computers          50.0  
5.2     websites           50.0  
        computers          50.0

In [19]:
def same_footing(c):
    dn = c.dom_new_peer.copy()
    pop = c.dom_tr.groupby("domain").user.nunique() / len(c.users)
    dn["rare"] = (dn.domain.map(pop).fillna(0) < 0.01).astype(float)
    d = pad_days(c, dn.groupby(["user", "day"])[["one", "rare", "s", "peer"]].sum().reset_index(), ["one", "rare", "s", "peer"])
    return [metrics(f"r{c.rel} web, naive new sites", robust_max(d, "one"), c), metrics(f"r{c.rel} web, rare new sites", robust_max(d, "rare"), c),
            metrics(f"r{c.rel} web, recommender surprise", robust_max(d, "s"), c), metrics(f"r{c.rel} web, recommender peer-of-item", robust_max(d, "peer"), c)]

for c in (c42, c52):
    display(table(same_footing(c)))

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
"r4.2 web, naive new sites",0.050,2.0,0.099,0.594,3/30,4.2/30,0/10
"r4.2 web, rare new sites",0.805,40.4,0.469,0.778,30/30,1.4/30,10/10
"r4.2 web, recommender surprise",0.000,5.0,0.122,0.713,1/30,8/30,4/10
"r4.2 web, recommender peer-of-item",0.450,17.0,0.364,0.839,22.6/30,3/30,9.6/10


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100,S4 top100
detector,,,,,,,,
"r5.2 web, naive new sites",0.100,3.8,0.072,0.589,0/29,2.5/30,2/10,0.5/30
"r5.2 web, rare new sites",0.349,11.4,0.120,0.643,5.2/29,0.9/30,6.7/10,1.3/30
"r5.2 web, recommender surprise",0.050,4.0,0.073,0.626,0/29,5/30,1/10,1/30
"r5.2 web, recommender peer-of-item",0.350,31.0,0.259,0.720,29/29,0/30,10/10,0/30


In [20]:
NOREC = [k for k in BASE + PW if k != "w_rare"]
ABL = {"no website novelty": NOREC,
       "+ naive website novelty (new + rare sites)": NOREC + ["n_dom", "w_rare"],
       "+ recommender (surprise + peer-of-item)": NOREC + ["a_dom", "dom_peer"],
       "+ both": NOREC + ["n_dom", "w_rare", "a_dom", "dom_peer"]}
for c in (c42, c52):
    rows, lrows = [], []
    for k, cols in ABL.items():
        rows.append(metrics(f"r{c.rel} LR: {k}", cv_score(c, cols), c)); lrows.append({"combiner": f"r{c.rel} LR: {k}", **loso(c, cols)})
    display(table(rows)); display(pd.DataFrame(lrows).set_index("combiner"))

QV = {"destination queue = recommender peer-of-item": QUEUES,
      "destination queue = rare sites (no recommender)": {**QUEUES, "unusual destination": "w_rare"},
      "destination queue = naive new sites": {**QUEUES, "unusual destination": "n_dom"},
      "no destination queue (3 queues share the 100)": {k: v for k, v in QUEUES.items() if k != "unusual destination"}}
for c in (c42, c52):
    picks = {name: queue_budget(c, 100, qm)[0] for name, qm in QV.items()}
    base3 = set(picks["no destination queue (3 queues share the 100)"])
    rows = []
    for name, ppl in picks.items():
        r = budget_people(f"r{c.rel} {name}", ppl, c); r["insiders the 3-queue version misses"] = sum(1 for u in ppl if u in c.lab and u not in base3); rows.append(r)
    display(pd.DataFrame(rows).set_index("method"))
print(f"total notebook runtime: {(time.time() - T0) / 60:.1f} min")

,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100
detector,,,,,,,
r4.2 LR: no website novelty,0.95,50.0,0.817,0.979,30/30,21/30,9/10
r4.2 LR: + naive website novelty (new + rare sites),1.00,51.0,0.848,0.983,30/30,23/30,9/10
r4.2 LR: + recommender (surprise + peer-of-item),1.00,52.0,0.853,0.984,30/30,23/30,9/10
r4.2 LR: + both,1.00,52.0,0.857,0.984,30/30,22/30,10/10


,S1 hidden,S2 hidden,S3 hidden
combiner,,,
r4.2 LR: no website novelty,30/30,13/30,10/10
r4.2 LR: + naive website novelty (new + rare sites),30/30,8/30,9/10
r4.2 LR: + recommender (surprise + peer-of-item),30/30,16/30,10/10
r4.2 LR: + both,30/30,14/30,10/10


,P@20,hits@70,AP,AUC,S1 top100,S2 top100,S3 top100,S4 top100
detector,,,,,,,,
r5.2 LR: no website novelty,0.80,46.0,0.553,0.943,27/29,11/30,5/10,7/30
r5.2 LR: + naive website novelty (new + rare sites),0.85,45.0,0.600,0.947,26/29,13/30,7/10,9/30
r5.2 LR: + recommender (surprise + peer-of-item),0.80,48.0,0.602,0.950,27/29,15/30,10/10,9/30
r5.2 LR: + both,0.90,49.0,0.630,0.950,27/29,16/30,10/10,9/30


,S1 hidden,S2 hidden,S3 hidden,S4 hidden
combiner,,,,
r5.2 LR: no website novelty,24/29,3/30,6/10,3/30
r5.2 LR: + naive website novelty (new + rare sites),22/29,5/30,7/10,2/30
r5.2 LR: + recommender (surprise + peer-of-item),21/29,3/30,10/10,2/30
r5.2 LR: + both,20/29,3/30,10/10,2/30


,people,insiders,S1,S2,S3,insiders the 3-queue version misses
method,,,,,,
r4.2 destination queue = recommender peer-of-item,100,42,29/30,4/30,9/10,3
r4.2 destination queue = rare sites (no recommender),100,44,30/30,5/30,9/10,4
r4.2 destination queue = naive new sites,100,34,28/30,4/30,2/10,0
r4.2 no destination queue (3 queues share the 100),100,41,30/30,4/30,7/10,0


,people,insiders,S1,S2,S3,S4,insiders the 3-queue version misses
method,,,,,,,
r5.2 destination queue = recommender peer-of-item,100,53,24/29,2/30,7/10,20/30,5
r5.2 destination queue = rare sites (no recommender),100,50,24/29,3/30,3/10,20/30,3
r5.2 destination queue = naive new sites,100,47,24/29,2/30,1/10,20/30,0
r5.2 no destination queue (3 queues share the 100),100,59,27/29,3/30,4/10,25/30,0


total notebook runtime: 5.4 min


**Finding.** The recommender's contribution is modest and conditional.

- **Its own job** (predicting normal staff's first-time site use): mean percentile rank is **35.9% vs 35.5% for "guess the popular sites"** on r4.2, and 35.8% vs 38.3% on r5.2. That's at the popularity baseline (50% is random). Its RMSE is *worse* than popularity, because implicit scores aren't calibrated probabilities. So RMSE is a diagnostic only.
- **On identical footing:** a plain **rare-new-site count beats peer-of-item on r4.2** (P@20 0.81 vs 0.45; AP 0.469 vs 0.364). **Peer-of-item wins on r5.2** (31 vs 11.4 of 99; AP 0.259 vs 0.120).
- **In the learned combiner:**
  - Replacing naive website features with recommender features: +1 (r4.2), +3 (r5.2).
  - *Adding* recommender features to naive ones on r5.2: 45 to 49, AP 0.600 to 0.630.
  - The hidden-scenario results are mixed.
- **As a queue, at 100 people:**
  - r4.2: recommender 42, rare sites 44, naive 34, no destination queue 41.
  - r5.2: recommender 53, rare sites 50, naive 47, **no destination queue 59**. The recommender queue found 5 insiders the 3-queue version missed, but displaced 11 others.

**Verdict:** keep the recommender as optional evidence and a tested research branch. Not as a queue, and not as the headline.

## Conclusions and the plan for the build

### What we learned

1. **Baselines first.** Personal baselines put 20 of 20 insiders at the top on r4.2.
2. **Own-history weekly percentiles** give the best top of the list on both releases (P@20 1.00). Their AP is the best single signal on r5.2 only.
3. **Snoopers are caught by persistence:** logging into unusual computers week after week, discounted for people who already roamed. With identical counting, ownership, naive and recommender versions all reach 27 of 30.
4. **The recommender's honest role is small.**
   - Its own predictive accuracy is at the popularity baseline.
   - It adds modest, conditional value in the combiner (r5.2: 45 to 49 when added to naive features).
   - As a queue it displaced more insiders than it added on r5.2 (53 vs 59).
   - Keep it as optional evidence ("nobody in your peer group visits this site"), not a pillar.
5. **Combining:**
   - Trained combiners don't generalize to unseen attack types.
   - Label-free family-first fusion beat queues on r4.2 (55 vs 42 at 100 reviews).
   - Queues beat fusion on r5.2 (53 vs 43), because snoopers need their own list.
6. **The external review mattered.** It found real bugs (an unbalanced hold-out test, a boundary-week leak, tie luck, a silently swapped comparator) and overturned three headline claims. Saying that openly is a strength in front of judges.

### The plan for the build (Oct 17-18, on LinuxONE)

| Stage | What | Status |
|---|---|---|
| 1. Load | PySpark (local mode) to daily, person-computer and person-site tables | tested |
| 2. Baselines | R1/R2 personal baselines as the bar to beat | tested |
| 3. Main ranked list | **Frozen family-first Stouffer fusion** (behavior, own-history percentiles, peers, rare sites, weekly LOF/AE, recommender family), with R2 and the percentile list as comparison views | tested on both releases; beat R2 at 100 reviews on both (55 vs 44, 43 vs 39) |
| 4. Out-of-role view | Weekly persistence of logins to unusual computers, with a roaming discount, shown as an **experimental investigation view** with no reserved review slots | r5.2 only; designed after seeing it |
| 5. Evidence | Per-person facts for Granite (feature contributions, timelines); the recommender's "nobody like you visits this" as an optional fact | recommender kept inside the tested fusion; its individual necessity not proven |
| 6. Explain and show | Granite cited explanations; dashboard; the honest evaluation table (baseline vs system, per scenario, unseen-attack test, equal budget) | not yet built; Granite reliability and 3-vCPU runtime untested |

This follows the external reviewer's final recommendation: one tested default ranking, plus an experimental view, instead of an untested allocation of queue slots.

### What changed from the original abstract, and why

| Original | Now | Reason (step) |
|---|---|---|
| Recommender as the core engine | Optional evidence; not a queue | At the popularity baseline on its own job; displaced insiders as a queue (15) |
| "Recommender forgives colleagues' PCs; ownership fixes it" | Persistence (weekly) catches snoopers; ownership adds precision only | Controlled ablation, 27 vs 27 vs 27 (13) |
| One combined risk score, then four queues | Frozen fusion as the default ranking, plus an experimental out-of-role view | Fusion beat R2 on both releases; queues won only on r5.2, and lost slots to the recommender queue (14, 15) |
| Trained combiner as the headline | Label-free signals as the headline | Unseen-attack test (7, 12) |

### Honest limits to state in the pitch

- **Retrospective, not real-time:** a user-level ranking over the whole test period; detection delay is not measured.
- **Job-hunters stay hard:** 0 to 16 of 30, depending on the method. Published work reports 12.5 to 74%.
- **Synthetic benchmark data:** labeled real insider logs are very rare. LANL releases real authentication logs with red-team labels, which would make a good third test.
- **Partly post-hoc design:** the out-of-role signal and the build plan were shaped after seeing r5.2, and need an untouched test.

## Platform notes for the LinuxONE build

From IBM's own repositories (`jupyter-lab-ml`, `ibm-zdnn-plugin`, IBM Z accelerated PyTorch and Snap ML):

- The event image ships TensorFlow with `ibm-zdnn-plugin`, PyTorch with `torch_nnpa`, XGBoost, scikit-learn and llama.cpp with Granite. It does **not** ship PySpark or pyarrow; s390x has no pyarrow wheel, so install OpenJDK 17 and PySpark and avoid pandas UDFs and Arrow conversions. Reading the Parquet files written here needs pyarrow, so on the server keep intermediate tables in Spark or write CSV.
- Run in `local[3]` (3 vCPUs) and set driver memory before Spark starts.
- To earn the Telum (AI accelerator) points: if an autoencoder is kept, run its scoring through Keras with `ibm-zdnn-plugin` or `torch_nnpa`; a boosted model can be trained in XGBoost and loaded into Snap ML. ALS, Isolation Forest, LOF and the rules stay on CPU; say so honestly.
- Granite explanations: reuse the loan-approval demo's big-endian model, give it an evidence pack with an ID per fact, require citations, check the numbers in code, and pre-generate the explanations for the video.

## Questions for the reviewer

1. **Protocol:** is user-level ranking with these train cut-offs and the leave-one-scenario-out test a fair evaluation, and is anything still leaking?
2. **Overfitting to r4.2:** many choices were made with r4.2's answer key in view. Which results would you trust only after a third test?
3. **The ownership rule** was designed after seeing ALS fail on scenario 4 in r5.2. Is it principled enough to present, and how would you test it independently (e.g., CERT r6.2 or LANL authentication logs)?
4. **Queues versus fusion:** is "specialist queues with a fixed review budget" the right framing, and how should queue sizes be set without labels?
5. **Scenario 2 (job-hunters)** stays hard. Is there a label-free signal we're missing?
6. **The recommender's role:** is "ALS for shared resources (websites), ownership for personal ones (computers)" a defensible story, or is the recommender now decorative?